## 11.Retrievers

<pre>
A vector store stores and searches vectors. A retriever is the component that gets relevant documents for a question.

</pre>

In [1]:
from langchain_community.document_loaders import PyPDFLoader

loader=PyPDFLoader("text.pdf")
documents=loader.load()
print(len(documents))

Ignoring wrong pointing object 43 0 (offset 0)
Ignoring wrong pointing object 72 0 (offset 0)
Ignoring wrong pointing object 73 0 (offset 0)


15


In [2]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings=HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')

c:\Users\DELL\OneDrive\Documents\Langchain\.langchainEnv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2521.78it/s]


In [5]:
from langchain_chroma import Chroma

chromaDb=Chroma.from_documents(documents=documents,embedding=embeddings,collection_name="Tottodoi")


In [6]:
# Now let's convert our existing Chroma vector store into a retriever.

retriever=chromaDb.as_retriever()
results=retriever.invoke("How were the members of the Constituent Assembly elected, and who elected them?")

In [7]:
for doc in results:
    print(doc.page_content)
    print(doc.metadata)
    print("-------------------------------------")

Contents
•Formation and Representation in the Constituent 
Assembly 
•Committees and Sub-Committees
{'producer': 'macOS Version 26.2 (Build 25C56) Quartz PDFContext', 'title': 'THe Making of Indian Constitution I', 'page': 1, 'creator': 'Keynote', 'total_pages': 15, 'page_label': '2', 'creationdate': "D:20260106092821Z00'00'", 'source': 'text.pdf', 'moddate': "D:20260106092821Z00'00'"}
-------------------------------------
Composition, Representation and Structure
•The membership of Congress in the Constituent Assembly and outside held social, economic and political 
views ranging from the reactionary to the revolutionary; the leaders of the Assembly were Independence 
movement leaders. 
•The majority in Constituent Assembly resulted from December 1945 provincial legislature elections and 
from Partition; The outcome of the elections in 1946, where Congress won majority, it filled 208 of the total 
296 allowed to provinces. It had a majority of 69 percent and after partition, it increa

In [8]:
from langchain_groq import ChatGroq

model=ChatGroq(model_name='openai/gpt-oss-20b')

In [9]:
context="\n\n".join(doc.page_content for doc in results)
print(context)

Contents
•Formation and Representation in the Constituent 
Assembly 
•Committees and Sub-Committees

Composition, Representation and Structure
•The membership of Congress in the Constituent Assembly and outside held social, economic and political 
views ranging from the reactionary to the revolutionary; the leaders of the Assembly were Independence 
movement leaders. 
•The majority in Constituent Assembly resulted from December 1945 provincial legislature elections and 
from Partition; The outcome of the elections in 1946, where Congress won majority, it filled 208 of the total 
296 allowed to provinces. It had a majority of 69 percent and after partition, it increased to 82 percent.   
•Initially, the total members were 389 (from British Indian Provinces and Princely States); After partition, 
the Assembly was reduced from 389 to 299 members. 
•The Assembly functioned as both a constitution-making body and later, as the provincial legislature of 
independent India - members were simul

In [13]:
# Pass the context and question to the LLM

from langchain_core.prompts import ChatPromptTemplate

prompt=ChatPromptTemplate.from_template("""
Answer the question using only the context below.
If the answer is not available in the context,say 'I couldn't find the answer in the provided document.'
Context:{context}
Question:{question}

""")

In [14]:
question="Based strictly on the context provided above, explain how the members of the Constituent Assembly were elected, and who exactly elected them?"

In [16]:
chain=prompt | model
response=chain.invoke({"context":context,"question":question})
print(response.content)

**Election method**

* The Constituent Assembly was not chosen by a direct popular vote.  
* Its members were **indirectly elected** by the members of the provincial legislatures that had been formed under the Government of India Act 1935.  
* The provincial legislatures themselves had a highly restricted franchise (property, education, tax qualifications), so the electorate for the Assembly was limited.  
* Seats in the Assembly were allocated on a **communal basis** (General, Muslim, Sikh, etc.) for the British Indian provinces.  
* Representatives from the princely states were **not elected**; they were **nominated** by the states.

**Who elected them?**

* The **members of each provincial legislature** chose the Assembly representatives.  
* In practice, the Congress high‑command let each provincial legislature select its own delegates, preserving the diversity already present in those legislatures.  
* Thus, the Assembly’s membership came from the provincial legislatures (indirect

### Multi-Query Retrieval

<pre>
1. Why do we need Multi-Query Retrieval?

Suppose your PDF contains information about RAG.

You ask:

What is RAG?

Your retriever searches for chunks relevant to that exact question. But what if the PDF explains RAG using different words, like “retrieval-augmented generation combines external knowledge with an LLM”?

A single search might miss some useful chunks because your question doesn't capture every possible way the topic is described.

Multi-query retrieval solves this by generating multiple versions of the same question and searching with each one.

2. How does it work?

Original question "What is RAG?"

LLM generates alternative queries

1. How does Retrieval-Augmented Generation work?

2. How do LLMs use external documents to answer questions?

3. What is the purpose of retrieving context for an LLM?

Retriever searches all queries

Each query can find different relevant chunks.

Combine retrieved chunks

Remove duplicates and pass the results as context to the LLM.
</pre>

In [ ]:
# This retrieves up to 3 chunks per search query
base_retriever=chromaDb.as_retriever(
    search_kwargs={"k":3}
)

In [18]:
from langchain_classic.retrievers import MultiQueryRetriever

multi_retriever=MultiQueryRetriever.from_llm(
    retriever=base_retriever,
    llm=model,
    include_original=True
)

Understand the parameters:

retriever=base_retriever: tells it how to search Chroma.

llm=model: uses Groq to generate alternative questions.

include_original=True: also searches using your original question.

In [19]:
# Make the generated questions visible

import logging

logging.basicConfig(level=logging.INFO)

logging.getLogger(
    "langchain_classic.retrievers.multi_query"
).setLevel(logging.INFO)

In [21]:
# ask a question

question="explain how the members of the Constituent Assembly were elected, and who exactly elected them?"
docs = multi_retriever.invoke(question)

INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
INFO:langchain_classic.retrievers.multi_query:Generated queries: ['How were the delegates chosen for the Constituent Assembly, and which body or group carried out the election?  ', 'What was the electoral mechanism that selected the members of the Constituent Assembly, and who held the authority to elect them?  ', 'Describe the process by which the Constituent Assembly’s members were elected, and identify the specific electorate responsible for that election.']


In [ ]:

# Inspect the retrieved chunks
print("Number of retrieved chunks:", len(docs))

for i, doc in enumerate(docs, start=1):
    print(f"\n--- Chunk {i} ---")
    print(doc.page_content)
    print("Metadata:", doc.metadata)

Number of retrieved chunks: 4

--- Chunk 1 ---
Contents
•Formation and Representation in the Constituent 
Assembly 
•Committees and Sub-Committees
Metadata: {'producer': 'macOS Version 26.2 (Build 25C56) Quartz PDFContext', 'page_label': '2', 'creator': 'Keynote', 'source': 'text.pdf', 'page': 1, 'total_pages': 15, 'creationdate': "D:20260106092821Z00'00'", 'title': 'THe Making of Indian Constitution I', 'moddate': "D:20260106092821Z00'00'"}

--- Chunk 2 ---
Composition, Representation and Structure
•The membership of Congress in the Constituent Assembly and outside held social, economic and political 
views ranging from the reactionary to the revolutionary; the leaders of the Assembly were Independence 
movement leaders. 
•The majority in Constituent Assembly resulted from December 1945 provincial legislature elections and 
from Partition; The outcome of the elections in 1946, where Congress won majority, it filled 208 of the total 
296 allowed to provinces. It had a majority of 69 pe

In [23]:
# Combine the chunks into context
context = "\n\n".join(
    doc.page_content for doc in docs
)

print(context)

Contents
•Formation and Representation in the Constituent 
Assembly 
•Committees and Sub-Committees

Composition, Representation and Structure
•The membership of Congress in the Constituent Assembly and outside held social, economic and political 
views ranging from the reactionary to the revolutionary; the leaders of the Assembly were Independence 
movement leaders. 
•The majority in Constituent Assembly resulted from December 1945 provincial legislature elections and 
from Partition; The outcome of the elections in 1946, where Congress won majority, it filled 208 of the total 
296 allowed to provinces. It had a majority of 69 percent and after partition, it increased to 82 percent.   
•Initially, the total members were 389 (from British Indian Provinces and Princely States); After partition, 
the Assembly was reduced from 389 to 299 members. 
•The Assembly functioned as both a constitution-making body and later, as the provincial legislature of 
independent India - members were simul

In [24]:
# pass the context to groq

from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template("""
Answer the question using only the provided context.

If the context does not contain the answer,
say you could not find it in the document.

Context:
{context}

Question:
{question}
""")

chain = prompt | model

response = chain.invoke({
    "context": context,
    "question": question
})

print(response.content)

INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


**How the Constituent Assembly members were chosen**

* The Assembly was **not elected directly by the people**.  
* Each member was selected **indirectly by the provincial legislatures** (the legislatures of the British‑Indian provinces).  
* Those provincial legislatures themselves were elected under the 1935 Government of India Act, which imposed a restricted franchise based on property, education and tax qualifications.  
* Seats in the Assembly were allocated on a communal basis (General, Muslim, Sikh, etc.) and the representatives from the princely states were **nominated** rather than elected.

**Who exactly elected them**

* The **members of the provincial legislatures**—the elected representatives of the provincial assemblies—were the ones who voted for the Constituent Assembly members.  
* In the princely states, the rulers or their representatives **nominated** the delegates.  

Thus, the Constituent Assembly was formed by indirect election: provincial legislature members ch

<pre>
Understand your complete pipeline

Your question
        |
Groq generates alternative questions
        |
Chroma searches for each question
        |
Using MiniLM embeddings
        |
Combine unique retrieved chunks
        |
Context + original question → Groq
        |
Final answer
</pre>